# ML Assignment 2 — Credit Card Default Classification

BITS Pilani WILP — Machine Learning

Five classifiers predict whether a credit-card client will **default next month**.
The numbers in this notebook match the README in the GitHub repository.


## 1. Problem

A bank wants to know, before next month, which clients are likely to miss a payment.
The target `default payment next month` is:

- `0` = no default
- `1` = default

This is a **binary classification** problem. The classes are not equal (about 22% default), so accuracy alone can look high even for a weak model.


## 2. Load the dataset

Source: [UCI Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)

Put `default_of_credit_card_clients.csv` (or `credit_card_default.csv`, or the original `.xls`) in the same folder as this notebook.
The ID column is only a row number, so it is dropped.


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

TARGET = "default payment next month"


def load_credit_data():
    csv_candidates = [
        Path("default_of_credit_card_clients.csv"),
        Path("credit_card_default.csv"),
    ]
    xls_path = Path("default of credit card clients.xls")
    csv_path = next((p for p in csv_candidates if p.exists()), None)
    if csv_path is not None:
        df = pd.read_csv(csv_path)
        print("Loaded", csv_path.resolve())
    elif xls_path.exists():
        # pandas + new xlrd cannot read this old .xls file, so use xlrd 1.2
        import xlrd
        book = xlrd.open_workbook(str(xls_path))
        sheet = book.sheet_by_index(0)
        columns = [sheet.cell_value(1, c) for c in range(sheet.ncols)]
        rows = [
            [sheet.cell_value(r, c) for c in range(sheet.ncols)]
            for r in range(2, sheet.nrows)
        ]
        df = pd.DataFrame(rows, columns=columns)
        df = df.apply(pd.to_numeric)
        print("Loaded", xls_path.resolve())
    else:
        raise FileNotFoundError(
            "Keep default_of_credit_card_clients.csv or 'default of credit card clients.xls' "
            "in the same folder as this notebook."
        )
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])
    return df


df = load_credit_data()
print("Rows, columns:", df.shape)
print("Default rate:", round(df[TARGET].mean(), 4))
df.head()


Loaded /workspace/artifacts/credit_card_default.csv
Rows, columns: (30000, 24)
Default rate: 0.2212


## 3. Features

23 features after dropping ID:

- `LIMIT_BAL`: credit limit
- `SEX`, `EDUCATION`, `MARRIAGE`, `AGE`
- `PAY_0`, `PAY_2` to `PAY_6`: repayment status for the last six months
- `BILL_AMT1` to `BILL_AMT6`: bill amounts
- `PAY_AMT1` to `PAY_AMT6`: amounts paid


In [2]:
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)
print("Features:", X.shape[1])
print(list(X.columns))
print(y.value_counts().rename({0: "no default", 1: "default"}))


Features: 23
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
default payment next month
no default    23364
default        6636
Name: count, dtype: int64


## 4. Train / test split

80% train (24,000) and 20% test (6,000), stratified so both parts keep the same default rate.
`random_state=42` keeps the split the same every time.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape[0], " Test:", X_test.shape[0])
print("Train default rate:", round(y_train.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))


Train: 24000  Test: 6000
Train default rate: 0.2212
Test default rate: 0.2212


## 5. Scaling

`StandardScaler` is fit on the **training** rows only, then applied to the test rows.
Logistic Regression, KNN, Naive Bayes, and SVM use the scaled values.
The Decision Tree uses the original values, because a tree does not need scaling.


In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Scaled train shape:", X_train_s.shape)


Scaled train shape: (24000, 23)


## 6. Metric function

For every model we record:

- Accuracy
- AUC (ROC-AUC, using the probability of default)
- Precision, Recall, F1 (positive class = default)
- MCC (Matthews correlation coefficient; fairer when classes are unequal)

A model that always says "no default" would score about **0.7788** accuracy and **0** recall. So accuracy near 0.81 is only a small gain.


In [5]:
def evaluate(name, y_true, y_pred, y_score):
    row = {
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "AUC": round(roc_auc_score(y_true, y_score), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_true, y_pred), 4),
    }
    print(name, row["Accuracy"], row["AUC"], row["Precision"], row["Recall"], row["F1"], row["MCC"])
    return row


results = []


## 7. Logistic Regression

A straight-line model. It uses the scaled features.


In [6]:
logistic = LogisticRegression(max_iter=2000, random_state=42)
logistic.fit(X_train_s, y_train)
results.append(
    evaluate(
        "Logistic Regression",
        y_test,
        logistic.predict(X_test_s),
        logistic.predict_proba(X_test_s)[:, 1],
    )
)


Logistic Regression 0.8077 0.7076 0.6868 0.2396 0.3553 0.3244


## 8. Decision Tree

Depth is limited (`max_depth=6`, `min_samples_leaf=50`) so the tree does not memorize rare rows.
This model uses the **unscaled** features.


In [7]:
tree = DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, random_state=42)
tree.fit(X_train, y_train)
results.append(
    evaluate(
        "Decision Tree",
        y_test,
        tree.predict(X_test),
        tree.predict_proba(X_test)[:, 1],
    )
)


Decision Tree 0.8167 0.7508 0.6697 0.3376 0.4489 0.3828


## 9. KNN

`k=15` neighbours on the scaled features. A larger k stops one odd neighbour from deciding the label.


In [8]:
knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train_s, y_train)
results.append(
    evaluate(
        "KNN",
        y_test,
        knn.predict(X_test_s),
        knn.predict_proba(X_test_s)[:, 1],
    )
)


KNN 0.8065 0.7325 0.6153 0.3338 0.4328 0.3507


## 10. Gaussian Naive Bayes

Assumes each feature is an independent bell curve. It uses the scaled features.
It often flags more clients as defaulters, so recall goes up and precision goes down.


In [9]:
nb = GaussianNB()
nb.fit(X_train_s, y_train)
results.append(
    evaluate(
        "Naive Bayes",
        y_test,
        nb.predict(X_test_s),
        nb.predict_proba(X_test_s)[:, 1],
    )
)


Naive Bayes 0.7525 0.7249 0.4515 0.5539 0.4975 0.3386


## 11. Support Vector Machine

A **linear** SVM (`LinearSVC`). `CalibratedClassifierCV` turns its scores into probabilities so AUC can be computed.
It uses the scaled features. A full non-linear SVM on 24,000 rows is very slow, and a linear boundary is enough for this comparison.


In [10]:
svm = CalibratedClassifierCV(
    LinearSVC(max_iter=5000, dual=False, random_state=42),
    cv=3,
)
svm.fit(X_train_s, y_train)
results.append(
    evaluate(
        "Support Vector Machine",
        y_test,
        svm.predict(X_test_s),
        svm.predict_proba(X_test_s)[:, 1],
    )
)


Support Vector Machine 0.8077 0.7044 0.6876 0.2389 0.3546 0.3242


## 12. Comparison table


In [11]:
comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))


         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.8077 0.7076     0.6868  0.2396 0.3553 0.3244
         Decision Tree    0.8167 0.7508     0.6697  0.3376 0.4489 0.3828
                   KNN    0.8065 0.7325     0.6153  0.3338 0.4328 0.3507
           Naive Bayes    0.7525 0.7249     0.4515  0.5539 0.4975 0.3386
Support Vector Machine    0.8077 0.7044     0.6876  0.2389 0.3546 0.3242


## 13. Observations

| ML Model Name | Observation |
|---|---|
| Logistic Regression | Accuracy 0.8077 is only a little above always saying "no default" (0.7788). Precision is high (0.6868) but recall is low (0.2396), so it misses most real defaulters. |
| Decision Tree | Best overall model on this split. Highest accuracy (0.8167), highest AUC (0.7508), and highest MCC (0.3828). |
| KNN | Accuracy 0.8065. AUC 0.7325 is better than Logistic Regression and SVM. Needs scaled features. |
| Naive Bayes | Lowest accuracy (0.7525) because it raises more false alarms. Best recall (0.5539) and best F1 (0.4975), so it catches more defaulters. |
| Support Vector Machine | Almost the same as Logistic Regression (accuracy 0.8077, AUC 0.7044, MCC 0.3242). |
| Overall, Winner for your dataset? | **Decision Tree**. If the bank cares more about catching defaulters than about false alarms, Naive Bayes is the stronger recall / F1 choice. |
